In [1]:
import tsl
import torch
import numpy as np
import pandas as pd
from tsl.datasets import MetrLA
from einops import rearrange
from torch_geometric.utils.undirected import is_undirected
from tsl.engines import Imputer, Predictor
from torch_geometric.utils.loop import remove_self_loops
from torch_geometric.utils.isolated import contains_isolated_nodes
from tsl.data import SpatioTemporalDataset
from torch_geometric.utils import to_dense_adj, to_scipy_sparse_matrix
from tsl.data.datamodule import (SpatioTemporalDataModule,
                                 TemporalSplitter)
from tsl.data.preprocessing import StandardScaler
from topomodelx.utils.sparse import from_sparse
from torch_sparse import SparseTensor
from torch_geometric.utils.sparse import to_edge_index
import toponetx as tnx
import networkx as nx
import torch
from torch_cluster import random_walk
import itertools
from utils.random_walk import uniform_random_walk, uniqueness
import torch.nn.functional as F
from tsl.nn.layers.recurrent.base import GraphGRUCellBase
from tsl.nn.blocks.encoders.recurrent.base import RNNBase
from tsl.nn.models import base_model
from tsl.metrics import numpy as numpy_metrics
from tsl.metrics import torch as torch_metrics
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from tsl.data.preprocessing import StandardScaler, RobustScaler
from pytorch_lightning import Trainer
import math
import gc
import torch.nn as nn

import random
import torch
import numpy as np
import os



def find_cliques_size_k(G, k):
    all_cliques = set()
    for clique in nx.find_cliques(G):
        if len(clique) == k:
            all_cliques.add(tuple(sorted(clique)))
        elif len(clique) > k:
            for mini_clique in itertools.combinations(clique, k):
                all_cliques.add(tuple(sorted(mini_clique)))
    return list(all_cliques)




def seed_everything(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True
        torch.set_float32_matmul_precision('medium')  # 'medium' favors performance over precision

        # Enable TF32 format which is optimized for Tensor Cores on Ampere+ GPUs
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    return seed

seed_everything(42)

42

In [2]:
class HO_Pre(SpatioTemporalDataset):
    def __init__(self, 
                 target,
                 mask=None,
                 connectivity=None,
                 covariates=None,
                 scalers=None,
                 window=24,
                 horizon=2,
                 stride=1,
                 sparse = True,
                 signed = True,
                 *args, **kwargs):
        
        super().__init__(target = target,
                        mask=mask,
                        connectivity=connectivity,
                        covariates=covariates,
                        scalers=scalers,
                        window=window,
                        horizon=horizon,
                        stride=stride,
                        *args, **kwargs)     
        # Now compute matrices after all properties are properly set up
        self.sparse = sparse
        self.signed = signed
        self.rw_edge_index = self.inter_order_rw_matrix()
    
    @property
    def graph(self):
        """Property that lazily creates and caches the graph."""
        # if self._graph is None:
        # tensor_graph = to_dense_adj(self.edge_index, max_num_nodes=self.n_nodes).squeeze(0)
        sparse_graph = to_scipy_sparse_matrix(self.edge_index, self.edge_weight, num_nodes=self.n_nodes)
        # array_graph = np.array(tensor_graph)
        # self._graph = nx.from_numpy_array(array_graph)
        self._graph = nx.from_scipy_sparse_array(sparse_graph)
        return self._graph
    
    @property
    def nodes(self):
        """Property that lazily creates and caches the nodes."""
        return list(self.graph.nodes)
        
    @property
    def triangles(self):
        """Property that lazily computes and caches the triangles."""
        # if self._triangles is None:
        self._triangles = find_cliques_size_k(self.graph, 3)
        return self._triangles
    
    @property
    def simplical_complex(self):
        """Property that lazily creates and caches the simplicial complex."""
        # if self._sc is None:
        edge_set = np.array(self.edge_index.T)
        self._sc = tnx.SimplicialComplex(self.nodes + list(edge_set) + self.triangles)
        return self._sc

    def create_edge_dict(self, edge_index, edge_weight):
        return {(edge_index[0, i].item(), edge_index[1, i].item()): 
                {"edge_feature": edge_weight[i].item()} 
                for i in range(edge_index.shape[1])}
    
    def inter_order_rw_matrix(self):
        simplical_complex = self.simplical_complex
        simplical_complex.set_simplex_attributes(self.create_edge_dict(self.edge_index, self.edge_weight))
        
        new_edge_feature = torch.Tensor(np.array(list(simplical_complex.get_simplex_attributes("edge_feature").values())))
        
        L0_up = simplical_complex.adjacency_matrix(rank=0, index=False, signed = False).todense().A
        L1_up = simplical_complex.adjacency_matrix(rank=1, index=False, signed = False).todense().A
        L1_down = simplical_complex.coadjacency_matrix(rank=1, index=False, signed = False).todense().A
        L1 = L1_up + L1_down
        L2_down = simplical_complex.coadjacency_matrix(rank=2, index=False, signed = False).todense().A

        L0_up = torch.tensor(L0_up, dtype=torch.float) if isinstance(L0_up, np.ndarray) else L0_up
        L1 = torch.tensor(L1, dtype=torch.float) if isinstance(L1, np.ndarray) else L1
        L2_down = torch.tensor(L2_down, dtype=torch.float) if isinstance(L2_down, np.ndarray) else L2_down

        B1 = simplical_complex.incidence_matrix(rank=1, index=False, signed = False).todense().A # nodes x edges
        B1 = torch.tensor(B1, dtype=torch.float) if isinstance(B1, np.ndarray) else B1
        B1T = B1.T
        
        B2 = simplical_complex.incidence_matrix(rank=2, index=False, signed = False).todense().A # edges x triangles
        B2 = torch.tensor(B2, dtype=torch.float) if isinstance(B2, np.ndarray) else B2
        B2T = B2.T

        new_face_feature = B2T @ new_edge_feature

        
        total_size = L0_up.shape[0] + L1.shape[0] + L2_down.shape[0]
        block_matrix = torch.zeros(total_size, total_size)

        # Fill the diagonal blocks
        block_matrix[:L0_up.shape[0], :L0_up.shape[1]] = L0_up
        block_matrix[L0_up.shape[0]:L0_up.shape[0]+L1.shape[0], L0_up.shape[1]:L0_up.shape[1]+L1.shape[1]] = L1
        block_matrix[L0_up.shape[0]+L1.shape[0]:, L0_up.shape[1]+L1.shape[1]:] = L2_down

        # Fill the upper off-diagonal blocks
        block_matrix[:L0_up.shape[0], L0_up.shape[1]:L0_up.shape[1]+L1.shape[0]] = B1
        block_matrix[L0_up.shape[0]:L0_up.shape[0]+L1.shape[0], L0_up.shape[1]+L1.shape[1]:] = B2

        # Fill the lower off-diagonal blocks
        block_matrix[L0_up.shape[0]:L0_up.shape[0]+L1.shape[0], :L0_up.shape[1]] = B1T
        block_matrix[L0_up.shape[0]+L1.shape[0]:, L0_up.shape[1]:L0_up.shape[1]+L1.shape[1]] = B2T
        

        sparse_block_matrix = SparseTensor.from_dense(block_matrix)

        rw_edge_index = to_edge_index(sparse_block_matrix)    

        return rw_edge_index[0], new_edge_feature, new_face_feature

    def get(self, item):
        sample = super().get(item)

        rw_edge_index, edge_feature, triangle_feature = self.rw_edge_index
        
        sample.input['rw_edge_index'] = rw_edge_index
        sample.input['edge_feature'] = edge_feature
        sample.input['triangle_feature'] = triangle_feature
        
        return sample

In [3]:
dataset = MetrLA(root='./data/metrla')

connectivity = dataset.get_connectivity(threshold=0.4,
                                        include_self=False,
                                        # normalize_axis=1,
                                        force_symmetric=False,
                                        layout="edge_index")

covariates = {'u': dataset.datetime_encoded('day').values}

torch_dataset = HO_Pre(target=dataset.dataframe(),
                                      connectivity=connectivity,
                                      mask=dataset.mask,
                                      covariates=covariates,
                                      horizon=12,
                                      window=12,
                                      stride=1)
print(torch_dataset)

HO_Pre(n_samples=34249, n_nodes=207, n_channels=1)


/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:98: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  date_range = pd.date_range(df.index[0], df.index[-1], freq='5T')
/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/tsl/datasets/metr_la.py:109: FutureWarning: The 'method' keyword in DataFrame.replace is deprecated and will be removed in a future version.
  df = df.replace(to_replace=0., method='ffill')


In [4]:
# Normalize data using mean and std computed over time and node dimensions
scalers = {'target': StandardScaler(axis=(0, 1))}

# Split data sequentially:
#   |------------ dataset -----------|
#   |--- train ---|- val -|-- test --|
splitter = TemporalSplitter(val_len=0.1, test_len=0.2)

dm = SpatioTemporalDataModule(
    dataset=torch_dataset,
    scalers=scalers,
    splitter=splitter,
    batch_size=32,
    workers = 4
)

dm.setup()
print(dm)

{Train dataloader: size=24648}
{Validation dataloader: size=2728}
{Test dataloader: size=6849}
{Predict dataloader: None}


In [5]:
for i,data in enumerate(dm.train_dataloader()):
    if i == 0:
        print(data)

StaticBatch(
  input=(x=[b=32, t=12, n=207, f=1], u=[b=32, t=12, f=2], edge_index=[2, e=622], edge_weight=[e=622], rw_edge_index=[2, 19218], edge_feature=[577], triangle_feature=[550]),
  target=(y=[b=32, t=12, n=207, f=1]),
  has_mask=True,
  transform=[x, y]
)


## Inter order random walk

In [6]:
import torch
from torch_cluster import random_walk


def uniform_random_walk(edge_index, nodes, batch_size, num_samples,timesteps,length):
    source, target = edge_index[0], edge_index[1]
    
    num_nodes = len(nodes)
    nodes = nodes.repeat(batch_size * num_samples * timesteps)

    walks, eids = random_walk(row=source,
                              col=target,
                              start=nodes,
                              walk_length=length,
                              return_edge_indices = True)
    
    walks = walks.view(batch_size, num_samples, timesteps, num_nodes, length+1)
    eids = eids.view(batch_size, num_samples, timesteps, num_nodes, length)
    
    return walks, eids


def uniqueness(walk):
    walk_equal = walk.unsqueeze(-1) == walk.unsqueeze(-2)
    # (1 * walk_equal) -- > bool to int such that can use argmax
    walk_equal = (1 * walk_equal).argmax(dim=-1)
    return walk_equal



## Model

In [7]:
class Embedding(nn.Module):
    def __init__(self, input_size,
                 hidden_size=32,
                 kernel_size=(4,1),
                 stride=(2,1),
                num_samples=5,
                rw_length=5):
        super(Embedding, self).__init__()
        self.kernel_size = kernel_size
        self.stride = stride
        self.conv = nn.Conv2d(
            in_channels=input_size, 
            out_channels=hidden_size, 
            kernel_size=kernel_size, 
            stride=stride
            )
        self.num_samples = num_samples
        self.rw_length = rw_length

    # def _gather_walk_features(self, features, walks):
    #     """Fully vectorized feature gathering without loops"""
    #     # Create batch indices tensor [batch_size, 1, 1, 1, 1]
    #     batch_indices = torch.arange(walks.shape[0], device=walks.device).view(-1, 1, 1, 1, 1)
    #     print(f"Memory 3: {torch.cuda.memory_allocated()/1e9:.2f} GB")
    #     # Expand batch_indices to match walks shape
    #     batch_indices = batch_indices.expand_as(walks)
    #     print(f"Memory 4: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    #     print('features',features.shape)
    #     print('walks',walks.shape)

    #     features torch.Size([64, 12, 1334, 1])
    #     walks torch.Size([64, 5, 12, 207, 6])
        
    #     # Gather features using advanced indexing
    #     # The result shape will be [batch_size, num_samples, num_nodes, length, feature_dim]
    #     return features[batch_indices, walks]

    def _gather_walk_features(self, features, walks):
        """Process on CPU then transfer back to GPU"""
        # Move tensors to CPU for processing
        cpu_features = features #--> batch_size, timestep, num_simplices, feature
        cpu_walks = walks #--> batch_size, num_random_walk_sample,timestep,num_nodes,random_walk_length

        # cpu_features torch.Size([64, 12, 1334, 1])
        # cpu_walks torch.Size([64, 5, 12, 207, 6])

        features = rearrange(cpu_features, 'b t n f -> b 1 t n 1 f')
        walks = rearrange(walks, 'b s t n l -> b s t n l 1')

        gathered_features = torch.gather(features.expand(-1, walks.shape[1], -1, -1, walks.shape[4], -1),
                                         3,
                                         walks.expand(-1, -1, -1, -1, -1, cpu_features.shape[3]))

        
        # Final shape: [batch_size, num_samples, timestep, num_nodes, length, feature_dim]
        # If you want to remove the timestep dimension:
        # output = rearrange(gathered_features, 'b s t n l f -> b s n l f')
        
        # Transfer back to GPU
        return gathered_features.to(walks.device)

    def forward(self,x,rw_edge_index, edge_feature,triangle_feature):
        # x --> [batch_size, time_steps, num_nodes, features]
        batch_size, T, num_nodes, features = x.shape

        edge_feature = edge_feature.unsqueeze(-1).repeat(batch_size,T,1,features)
        triangle_feature = triangle_feature.unsqueeze(-1).repeat(batch_size,T,1,features)

        x_new = torch.cat([x,edge_feature,triangle_feature], dim = -2) # --> [batch_size,time_steps,num_(node + edge + triangle),feature]
        # print(f"Memory 1: {torch.cuda.memory_allocated()/1e9:.2f} GB")  
        walks, eids = uniform_random_walk(
            edge_index=rw_edge_index.to('cuda'), 
            nodes=torch.arange(num_nodes).to('cuda'), 
            batch_size = batch_size,
            num_samples=self.num_samples,
            timesteps = T,
            length=self.rw_length
        ) # -- > [batch_size, num_samples, timesteps, num_nodes, length]
        uniqueness_walk = uniqueness(walks)
        walks, uniqueness_walk = walks.flip(-1), uniqueness_walk.flip(-1)
        uniqueness_walk = uniqueness_walk / uniqueness_walk.shape[-1]
        uniqueness_walk = uniqueness_walk * math.pi * 2.0
        uniqueness_walk = torch.cat(
            [
                uniqueness_walk.sin().unsqueeze(-1),
                uniqueness_walk.cos().unsqueeze(-1),
            ],
            dim=-1,
        )

        # Gather features for each node in the walks
        # [batch_size, num_samples, timesteps, num_nodes, length, feature_dim]
        gathered_features = self._gather_walk_features(x_new, walks)  

        # [batch_size, num_samples, timesteps, num_nodes, length, 1, feature_dim]
        # print(f"Memory 2: {torch.cuda.memory_allocated()/1e9:.2f} GB")  
        gathered_features = torch.concat([gathered_features, uniqueness_walk], dim = -1)
        gathered_features = gathered_features.unsqueeze(-2)
        batch_size, num_samples, timesteps, num_nodes, length, _, feature_dim = gathered_features.shape
        # print(f"Memory 3: {torch.cuda.memory_allocated()/1e9:.2f} GB")  
        

        # [batch_size, num_samples, timesteps, num_nodes, length, 1, feature_dim] 
        # -> [batch_size * num_samples * num_nodes * feature_dim, 1, timesteps, length] 
        gathered_features = rearrange(gathered_features, 'b s t n l d f -> (b s n f) d t l')
        gathered_features = F.pad(gathered_features,
                          pad=(0, self.kernel_size[1]-self.stride[1],
                               0, self.kernel_size[0]-self.stride[0]),
                          mode='replicate')
        # print(f"Memory 4: {torch.cuda.memory_allocated()/1e9:.2f} GB")  

        x_emb = self.conv(gathered_features)# -> [batch_size * num_samples * num_nodes * feature_dim, D, timesteps_reduce, length_reduce]
        
        x_emb = rearrange(x_emb, '(b s n f) d t l -> b s n f d t l',
                          b=batch_size, s=num_samples, n=num_nodes, f=feature_dim)
        # [batch_size * num_samples * num_nodes * feature_dim, D, timesteps_reduce, length_reduce] 
        # -> [batch_size , num_samples , num_nodes , feature_dim, D, timesteps_reduce, length_reduce]
        # print(f"Memory 5: {torch.cuda.memory_allocated()/1e9:.2f} GB")  
        return x_emb

In [8]:
class ModernTCNBlock(nn.Module):
    def __init__(self, num_nodes, M, D, kernel_size=(4,2), r=1.):
        super(ModernTCNBlock, self).__init__()
        self.dw_conv = nn.Conv2d(
            in_channels=M*D, 
            out_channels=M*D, 
            kernel_size=kernel_size,
            groups=M*D,
            padding='same'
            )  
        self.bn = nn.InstanceNorm2d(M*D)
        self.pw_con1 = nn.Conv2d(
            in_channels=M*D, 
            out_channels=M*D, 
            kernel_size=1,
            groups=M,
            # padding='same'
            )
        self.pw_con2 = nn.Conv2d(
            in_channels=M*D, 
            out_channels=M*D, 
            kernel_size=1,
            groups=D,
            # padding='same'
            )

    def forward(self, x_emb):
        # x_emb -> [batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce, length_reduce]
        batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce, length_reduce = x_emb.shape
        
        # First rearrangement
        x = rearrange(x_emb, 'b s n f d t l -> (b s n) (f d) t l')
        
        # Apply convolution
        x = self.dw_conv(x)
        x = F.gelu(self.bn(x))
        
        x = F.gelu(self.pw_con1(x))
        
        x = rearrange(x, '(b s n) (f d) t l -> (b s) n f d t l',
                      b=batch_size, s=num_samples, 
                      n=num_nodes, f=feature_dim, d=D)
        
        x = rearrange(x, '(b s) n f d t l -> (b s n) (d f) t l',
                      b=batch_size, s=num_samples)
        x = F.gelu(self.pw_con2(x))
        
        x = rearrange(x, '(b s n) (d f) t l -> b s d n f t l',
                      b=batch_size, s=num_samples, d=D,
                      n=num_nodes, f=feature_dim)
        
        # [batch_size, num_samples, num_nodes, feature_dim, D, timesteps_reduce, length_reduce]
        x = rearrange(x, 'b s d n f t l -> b s n f d t l')
        
        # residual connection
        out = x + x_emb
        
        return out

In [9]:
from einops.layers.torch import Rearrange
class ModernTCN(nn.Module):
    def __init__(self, input_size, num_nodes,windows, horizon,rw_sample, rw_length, hidden_size=32, kernel_size=(5,1), stride=(1,1), r=1, num_layers=2):
        super().__init__()
        self.num_layers = num_layers
        windows_patches = windows // stride[0]
        rw_length_patches = (rw_length+1) // stride[1]
        
        self.embed_layer = Embedding(input_size, hidden_size,kernel_size = kernel_size,
                                     stride = stride,num_samples=rw_sample,rw_length=rw_length)

        self.backbone = nn.ModuleList([ModernTCNBlock(num_nodes=num_nodes, M=input_size+2, D=hidden_size, kernel_size=kernel_size, r=r) for _ in range(num_layers)])

        
        # self.head = nn.Linear((input_size+2)*hidden_size*windows_patches*rw_length_patches, horizon)
        # self.head = nn.Sequential(
        #     nn.Linear((input_size+2)*hidden_size*windows_patches*rw_length_patches, hidden_size),
        #     nn.GELU(),
        #     nn.Linear(hidden_size, horizon)
            
        # )

        out_channels = (hidden_size // rw_sample) * rw_sample  # Make divisible

        self.head = nn.Sequential(
            nn.Conv1d(rw_sample*(input_size+2)*hidden_size*windows_patches*rw_length_patches, out_channels,
                      kernel_size=1,
                      groups=rw_sample),
            nn.GELU(),
            Rearrange('b f n -> b n f'),
            nn.Linear(out_channels, horizon)
            
        )

    def forward(self, x, rw_edge_index, edge_feature, triangle_feature):
        # x --> [batch_size, time_steps, num_nodes, features]
        x_emb = self.embed_layer(x, rw_edge_index, edge_feature, triangle_feature)

        for i in range(self.num_layers):
            x_emb = self.backbone[i](x_emb)


        # Flatten
        # [batch_size , num_samples , num_nodes , feature_dim, D, timesteps_reduce, length_reduce]
        # x_emb = x_emb.mean(1) # [batch_size , num_nodes , feature_dim, D, timesteps_reduce, length_reduce]
        z = rearrange(x_emb, 'b s n f d t l -> b (s f d t l) n')
        pred = self.head(z)

        # pred = rearrange(pred, 'b n f h -> b h n f')
        pred = rearrange(pred, 'b n h -> b h n')

        # RuntimeError: Predictions and targets are expected to have the same shape, 
        # but got torch.Size([64, 207, 1, 12]) and torch.Size([64, 12, 207, 1]).
        # return pred[:,:,:,0].unsqueeze(-1)
        return pred.unsqueeze(-1)
    

In [10]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
    'mae': torch_metrics.MaskedMAE(),
    'mse': torch_metrics.MaskedMSE(),
    'mae_step_1': torch_metrics.MaskedMAE(at=0),
   'mae_step_2': torch_metrics.MaskedMAE(at=2),
   'mae_step_3': torch_metrics.MaskedMAE(at=4),
   'mae_step_4': torch_metrics.MaskedMAE(at=6)
}


model = ModernTCN(input_size=1,hidden_size = 32,num_nodes=207, windows=12, horizon=12, rw_sample=5, rw_length=5,num_layers=3)

In [11]:
from torch.optim.lr_scheduler import MultiStepLR
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  'weight_decay':1e-3
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[40, 80, 120]}
)
# 'momentum':0.9,
#                  'nesterov':True

In [12]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping


checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=30,
        mode='min'
    )

from pytorch_lightning.profilers import PyTorchProfiler,AdvancedProfiler

from pytorch_lightning.profilers import AdvancedProfiler


# Configure a profiler focused on GPU memory usage and performance bottlenecks
profiler = PyTorchProfiler(
    on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/profiler'),
    # Record shapes of operator inputs
    record_shapes=True,
    # Profile CPU and CUDA operations
    profile_memory=True,
    # Use CUDA events for more accurate timing
    with_stack=True,
    # Activities to profile
    with_modules=True,    # Track module hierarchy
    activities=[
        torch.profiler.ProfilerActivity.CPU,
        torch.profiler.ProfilerActivity.CUDA,
    ],
    schedule=torch.profiler.schedule(
            wait=1,  # 前1步不采样
            warmup=1,  # 第2步作为热身，不计入结果
            active=3,  # 采集后面3步的性能数据
            repeat=2) # 重复2轮
    
)


# Create an Advanced Profiler instance
# profiler = AdvancedProfiler(
#     dirpath="log/profiler",     # Directory to save profiler outputs
#     filename="detailed_profile" # Base filename for the output
# )




trainer = Trainer(
        max_epochs=50,
    limit_train_batches = 150,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        devices=1,
        gradient_clip_val=5,
       callbacks=[checkpoint_callback, early_stop_callback],
      default_root_dir="logs",
    profiler=profiler,
    precision = '16',
    check_val_every_n_epoch = 10
    
)

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/lightning_fabric/connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [13]:
trainer.fit(predictor, datamodule=dm)

/netfs/tsp/student/2022/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /netfs/tsp/student/2022/zhu/ST_RUM/logs exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | ModernTCN        | 220 K  | train
-----------------------------------------------------------
220 K     Trainable params
0         Non-trainable params
220 K     Total params
0.881     Total estimated model params size (MB)
46        Modules in train mode
0         Modules in eval mode


Training: |                                                                                 | 0/? [00:00<?, ?i…

Arguments ['u', 'edge_index', 'edge_weight'] are filtered out. Only args ['rw_edge_index', 'edge_feature', 'triangle_feature', 'x'] are forwarded to the model (ModernTCN).


Validation: |                                                                               | 0/? [00:00<?, ?i…

Validation: |                                                                               | 0/? [00:00<?, ?i…

Validation: |                                                                               | 0/? [00:00<?, ?i…

Validation: |                                                                               | 0/? [00:00<?, ?i…

Validation: |                                                                               | 0/? [00:00<?, ?i…

`Trainer.fit` stopped: `max_epochs=50` reached.
FIT Profiler Report
Profile stats for: records
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                  [pl][module]__main__.ModernTCN: model         0.09%       1.606ms         2.92%      54.758ms       9.126ms       0.000us  

In [15]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /netfs/tsp/student/2022/zhu/ST_RUM/logs/epoch=49-step=7500.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
Loaded model weights from the checkpoint at /netfs/tsp/student/2022/zhu/ST_RUM/logs/epoch=49-step=7500.ckpt


Testing: |                                                                                  | 0/? [00:00<?, ?i…

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │     3.636526584625244     │
│         test_mae          │    3.8352200984954834     │
│      test_mae_step_1      │     2.477914810180664     │
│      test_mae_step_2      │    3.1141152381896973     │
│      test_mae_step_3      │    3.5846805572509766     │
│      test_mae_step_4      │     4.007406234741211     │
│         test_mse          │     59.67033386230469     │
└───────────────────────────┴───────────────────────────┘

TEST Profiler Report
Profile stats for: records
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                          ProfilerStep*         1.60%       6.559ms        99.99%     410.091ms     136.697ms       0.000us         0.00%     343.398ms     114.466ms       

[{'test_mae': 3.8352200984954834,
  'test_mae_step_1': 2.477914810180664,
  'test_mae_step_2': 3.1141152381896973,
  'test_mae_step_3': 3.5846805572509766,
  'test_mae_step_4': 4.007406234741211,
  'test_mse': 59.67033386230469,
  'test_loss': 3.636526584625244}]

In [ ]:
print(profiler.summary())

In [ ]:
cpu_features torch.Size([64, 12, 1334, 1]) --> batch_size, timestep, num_simplices, feature
cpu_walks torch.Size([64, 5, 12, 207, 6])  --> batch_size, num_random_walk_sample,timestep,num_nodes,random_walk_length

In [25]:
torch.rand([64, 12, 1334, 3])[:,:,:,0].unsqueeze(-1).shape

torch.Size([64, 12, 1334, 1])

In [6]:
class custom_GRU(torch.nn.GRU):
    def __init__(self, *args, **kwargs):
        kwargs["batch_first"] = True
        # kwargs["bidirectional"] = True
        super().__init__(*args, **kwargs)
    
    def forward(self, input, h_0):
        # input -- > [batch_size, num_samples, num_nodes, RW_length, features]
        # h0 -- > [num_layer*Direction(1 or 2), batch_size, num_samples, num_nodes, features]

        # input torch.Size([64, 10, 207, 4, 2])
        # h_0 torch.Size([1, 64, 10, 207, 64])
        
        num_direction = 2 if self.bidirectional else 1
        batch_shape = input.shape[:-2] #--> [batch_size, num_samples, num_nodes] 
        event_shape_input = input.shape[-2:] #--> [RW_length, features] 
        event_shape_h_0 = h_0.shape[-1]
        input = input.view(-1, *event_shape_input) #--> [batch_size * num_samples * num_nodes, RW_length, features]
        h_0 = h_0.view(num_direction * self.num_layers, -1, event_shape_h_0) #--> [num_layer*Direction(1 or 2), batch_size * num_samples * num_nodes, features]
        
        output, h_n = super().forward(input, h_0)
        output = output.view(*batch_shape, *output.shape[-2:])
        h_n = h_n.view(num_direction * self.num_layers, *batch_shape, *h_n.shape[-1:])
        return output, h_n

In [7]:
class Simplicial_RUMLayer(torch.nn.Module):
    def __init__(self,
                 input_size, 
                 output_size,
                 num_samples=1,
                 length=3,
                 *args,
                 **kwargs):
        super().__init__()
        self.num_samples = num_samples
        self.length = length
        self.output_size = output_size
        self.walkRNN = custom_GRU(2, output_size, bidirectional=True,*args,**kwargs)
        
        self.semanticRNN = custom_GRU(input_size + 2*output_size,
                                      output_size, *args,**kwargs)

        self.fc_edge = nn.Sequential(
            nn.Linear(1, input_size),
            nn.ReLU()
        )
        self.fc_triangle = nn.Sequential(
            nn.Linear(1, input_size),
            nn.ReLU()
        )

    def _gather_walk_features(self, features, walks):
        """Fully vectorized feature gathering without loops"""
        # Create batch indices tensor [batch_size, 1, 1, 1]
        batch_indices = torch.arange(walks.shape[0], device=walks.device).view(-1, 1, 1, 1)
        
        # Expand batch_indices to match walks shape
        batch_indices = batch_indices.expand_as(walks)
        
        # Gather features using advanced indexing
        # The result shape will be [batch_size, num_samples, num_nodes, length, feature_dim]
        return features[batch_indices, walks]

    def forward(self,x , rw_edge_index, edge_feature, triangle_feature):
        # x --> [batch_size, num_nodes, features]
        batch_size, num_nodes, features = x.shape

        print(f"Memory 1: {torch.cuda.memory_allocated()/1e9:.2f} GB")

        edge_feature = edge_feature.unsqueeze(-1).repeat(batch_size,1,1)
        triangle_feature = triangle_feature.unsqueeze(-1).repeat(batch_size,1,1)
    
        x_new = torch.cat([x, self.fc_edge(edge_feature), self.fc_triangle(triangle_feature)], dim = 1) # --> [batch_size, num_(node + edge + triangle), feature]
        
        print(f"Memory 2: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        with torch.no_grad():
            walks, eids = uniform_random_walk(
                edge_index=rw_edge_index.to('cuda'), 
                nodes=torch.arange(num_nodes).to('cuda'), 
                batch_size = batch_size,
                num_samples=self.num_samples,
                length=self.length
            ) # -- > [batch_size, num_samples, num_nodes, length]
            uniqueness_walk = uniqueness(walks)
            walks, uniqueness_walk = walks.flip(-1), uniqueness_walk.flip(-1)
            uniqueness_walk = uniqueness_walk / uniqueness_walk.shape[-1]
            uniqueness_walk = uniqueness_walk * math.pi * 2.0
            uniqueness_walk = torch.cat(
                [
                    uniqueness_walk.sin().unsqueeze(-1),
                    uniqueness_walk.cos().unsqueeze(-1),
                ],
                dim=-1,
            )

            # Gather features for each node in the walks
            # [batch_size, num_samples, num_nodes, length, feature_dim]
            gathered_features = self._gather_walk_features(x_new, walks)
            
        # print(f"Memory 3: {torch.cuda.memory_allocated()/1e9:.2f} GB")      
        h0 = torch.zeros(self.walkRNN.num_layers*2, *gathered_features.shape[:-2],
                         self.output_size, device=gathered_features.device)

        y_walk, h_walk = self.walkRNN(uniqueness_walk.to(h0.device), h0)
        
        h_walk = h_walk.mean(0, keepdim=True)

        # print(f"Memory 4: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        

        if self.semanticRNN.num_layers > 1:
            h_walk = h_walk.repeat(self.semanticRNN.num_layers, 1, 1, 1, 1)
            gathered_features = torch.cat([gathered_features, y_walk], dim=-1)
        else:
            gathered_features = torch.cat([gathered_features, y_walk], dim=-1)
            
        y, h = self.semanticRNN(gathered_features, h_walk)
        y = F.relu(y)
        y = y.mean(1)

        
        # del gathered_features,x_new,y_walk,h_walk,h,h0,uniqueness_walk
        # # gc.collect()
        # torch.cuda.empty_cache()
        # print(f"Memory 5: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        # # First collect Python garbage
        # gc.collect()
        # # Then clear CUDA cache
        # torch.cuda.empty_cache()
        return y[:, :, -1, :]

In [8]:
class ST_RUM_Cell(GraphGRUCellBase):
    def __init__(self,
                 input_size,
                 hidden_size):
        # instantiate gates
        forget_gate = Simplicial_RUMLayer(input_size+hidden_size,hidden_size)
        
        update_gate = Simplicial_RUMLayer(input_size+hidden_size,hidden_size)
        
        candidate_gate = Simplicial_RUMLayer(input_size+hidden_size,hidden_size)
        
        super().__init__(hidden_size=hidden_size,
                        forget_gate=forget_gate,
                        update_gate=update_gate,
                        candidate_gate=candidate_gate)

In [9]:
class ST_RUM(RNNBase):
    def __init__(self,
                 input_size: int,
                 hidden_size: int,
                 n_layers: int = 1,
                 cat_states_layers: bool = False,
                 return_only_last_state: bool = False):
        self.input_size = input_size
        self.hidden_size = hidden_size
        rnn_cells = [
            ST_RUM_Cell(input_size if i == 0 else hidden_size,
                      hidden_size) for i in range(n_layers)
        ]
        super().__init__(rnn_cells,cat_states_layers,return_only_last_state)

In [10]:
from torch import Tensor, nn
from einops import rearrange
from tsl.nn.blocks.decoders import MLPDecoder
from tsl.nn.models.base_model import BaseModel
from tsl.nn.blocks.encoders.conditional import ConditionalBlock


class ST_RUM_Model(BaseModel):
    return_type = Tensor
    def __init__(self,
                 input_size: int,
                 output_size: int,
                 horizon: int,
                 exog_size: int = 0,
                 hidden_size: int = 32,
                 ff_size: int = 256,
                 n_layers: int = 1,
                 dropout: float = 0.,
                 activation: str = 'relu'):
        super().__init__()
        if exog_size:
            self.input_encoder = ConditionalBlock(input_size=input_size,
                                                  exog_size=exog_size,
                                                  output_size=hidden_size,
                                                  activation=activation)
        else:
            self.input_encoder = nn.Linear(input_size, hidden_size)

        print(f"Memory before: {torch.cuda.memory_allocated()/1e9:.2f} GB")
        self.st_rum = ST_RUM(input_size=hidden_size,
                           hidden_size=hidden_size,
                           n_layers=n_layers,
                           return_only_last_state=True)
        print(f"Memory After: {torch.cuda.memory_allocated()/1e9:.2f} GB")

        

        self.readout = MLPDecoder(input_size=hidden_size,
                                  hidden_size=ff_size,
                                  output_size=output_size,
                                  horizon=horizon,
                                  activation=activation,
                                  dropout=dropout)

    def forward(self,
                x,
                rw_edge_index,
                edge_feature,
                triangle_feature,
                u):
        if u is not None:
            if u.dim() == 3:
                u = rearrange(u, 'b s c -> b s 1 c')
            x = self.input_encoder(x, u)
        else:
            x = self.input_encoder(x)
        out = self.st_rum(x,
                        rw_edge_index,
                        edge_feature,
                        triangle_feature)
        return self.readout(out)

In [11]:
loss_fn = torch_metrics.MaskedMAE()
# loss_fn = nn.L1Loss()
log_metrics = {
    'mae': torch_metrics.MaskedMAE(),
    'mse': torch_metrics.MaskedMSE(),
    'mae_step_1': torch_metrics.MaskedMAE(at=0),
   'mae_step_2': torch_metrics.MaskedMAE(at=2),
   'mae_step_3': torch_metrics.MaskedMAE(at=4),
   'mae_step_4': torch_metrics.MaskedMAE(at=6)
}

# model = models.DCRNNModel(input_size=1,exog_size=2, hidden_size = 64, output_size=1,
#                           horizon=7, ff_size = 128, dropout = 0.1,
#                           cache_support=True, n_layers = 1)

model = ST_RUM_Model(input_size=1,exog_size=2, hidden_size = 32, output_size=1,
                      horizon=12, ff_size = 16, dropout = 0.1,n_layers = 1)

Memory before: 0.00 GB
Memory After: 0.00 GB


In [12]:
from torch.optim.lr_scheduler import MultiStepLR
predictor = Predictor(
    model=model,                   # our initialized model
    optim_class=torch.optim.Adam,  # specify optimizer to be used...
    optim_kwargs={'lr': 1e-3,
                  'weight_decay':1e-3,
                 #  'momentum':0.9,
                 # 'nesterov':True
                 },    # ...and parameters for its initialization
    loss_fn=loss_fn,               # which loss function to be used
    metrics=log_metrics,                # metrics to be logged during train/val/test
    scale_target = False,
    # scheduler_class = MultiStepLR,
    # scheduler_kwargs = {'milestones':[40, 80, 120]}
)
# 'momentum':0.9,
#                  'nesterov':True

In [13]:
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping


checkpoint_callback = ModelCheckpoint(
    dirpath='logs',
    save_top_k=1,
    monitor='val_mae',
    mode='min',
)

early_stop_callback = EarlyStopping(
        monitor='val_mae',
        patience=30,
        mode='min'
    )



trainer = Trainer(
        max_epochs=300,
    # limit_train_batches = 32,
       # default_root_dir=cfg.run.dir,
        #logger=exp_logger,
        accelerator='gpu' if torch.cuda.is_available() else 'cpu',
        num_sanity_val_steps=0,
        #devices=1,
        gradient_clip_val=5,
       callbacks=[checkpoint_callback, early_stop_callback],
      default_root_dir="logs"
    
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Memory 1: 0.06 GB
Memory 2: 0.10 GB
Memory 3: 0.10 GB
Memory 4: 1.51 GB
Memory 5: 2.35 GB
Memory 1: 2.34 GB
Memory 2: 2.38 GB
Memory 3: 2.38 GB
Memory 4: 3.79 GB

In [14]:
trainer.fit(predictor, datamodule=dm)

/home/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:654: Checkpoint directory /home/zhu/high_order_imputation/ST_RUM/logs exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name          | Type             | Params | Mode 
-----------------------------------------------------------
0 | loss_fn       | MaskedMAE        | 0      | train
1 | train_metrics | MetricCollection | 0      | train
2 | val_metrics   | MetricCollection | 0      | train
3 | test_metrics  | MetricCollection | 0      | train
4 | model         | ST_RUM_Model     | 71.1 K | train
-----------------------------------------------------------
71.1 K    Trainable params
0         Non-trainable params
71.1 K    Total params
0.285     Total estimated model params size (MB)
69        Modules in train mode
0         Modules in eval mode
/home/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The '

Training: |                                                                                                   …

Arguments ['edge_index', 'edge_weight'] are filtered out. Only args ['rw_edge_index', 'triangle_feature', 'x', 'u', 'edge_feature'] are forwarded to the model (ST_RUM_Model).


Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …


Detected KeyboardInterrupt, attempting graceful shutdown ...

KeyboardInterrupt



In [15]:
predictor.freeze()

trainer.test(ckpt_path="best", dataloaders=dm.test_dataloader())

Restoring states from the checkpoint path at /home/zhu/high_order_imputation/ST_RUM/logs/epoch=3-step=1540.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at /home/zhu/high_order_imputation/ST_RUM/logs/epoch=3-step=1540.ckpt
/home/zhu/.conda/envs/grin/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


Testing: |                                                                                                    …

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │     3.710179567337036     │
│         test_mae          │     3.855433940887451     │
│      test_mae_step_1      │     2.589268445968628     │
│      test_mae_step_2      │    3.1744470596313477     │
│      test_mae_step_3      │    3.6122019290924072     │
│      test_mae_step_4      │     4.011588096618652     │
│         test_mse          │     57.3726692199707      │
└───────────────────────────┴───────────────────────────┘

[{'test_mae': 3.855433940887451,
  'test_mae_step_1': 2.589268445968628,
  'test_mae_step_2': 3.1744470596313477,
  'test_mae_step_3': 3.6122019290924072,
  'test_mae_step_4': 4.011588096618652,
  'test_mse': 57.3726692199707,
  'test_loss': 3.710179567337036}]

In [51]:
import torch

# Example tensors
batch_size = 2
num_elements = 10
feature_dim = 3
num_samples = 4
num_nodes = 5
length = 3

# Features tensor: [batch_size, num_elements, feature_dim]
features = torch.randn(batch_size, num_elements, feature_dim)

# Walks tensor: [batch_size, num_samples, num_nodes, length]
# Each value is an index into the elements dimension (values 0-9)
walks = torch.randint(0, num_elements, (batch_size, num_samples, num_nodes, length))

def gather_walk_features(features, walks):
    """Fully vectorized feature gathering without loops"""
    # Create batch indices tensor [batch_size, 1, 1, 1]
    batch_indices = torch.arange(walks.shape[0], device=walks.device).view(-1, 1, 1, 1)
    
    # Expand batch_indices to match walks shape
    batch_indices = batch_indices.expand_as(walks)
    # print('batch_indices',batch_indices)
    
    # Gather features using advanced indexing
    # The result shape will be [batch_size, num_samples, num_nodes, length, feature_dim]
    print("features",features.shape)
    print("batch_indices",batch_indices.shape)
    print("walks",walks.shape)
    
    return features[batch_indices, walks]

# Get result 
result = gather_walk_features(features, walks)
print("result", result.shape) 

# Verify with a manual approach for one element
b, s, n, l = 0, 1, 2, 1  # Example indices
manual_lookup = features[0, walks[0, 1, 2, 1]]

# Check if equal
torch.allclose(result[0, 1, 2, 1], manual_lookup)  # Should be True

features torch.Size([2, 10, 3])
batch_indices torch.Size([2, 4, 5, 3])
walks torch.Size([2, 4, 5, 3])
result torch.Size([2, 4, 5, 3, 3])


True

In [58]:
indexing = torch.tensor([[0,1,2],[0,1,2]])
indexing

tensor([[0, 1, 2],
        [0, 1, 2]])

In [59]:
values = torch.tensor([0,3,1,4])

In [60]:
values[indexing]

tensor([[0, 3, 1],
        [0, 3, 1]])

In [ ]:
F.silu